# Single-Experiment Validation

Detailed validation of a single feature-group combination for FL_UDSD classification.

**Set your config in the next cell**, then run all. You'll get per-fold confusion matrices, per-class precision/recall/F1, group-leakage checks, and saved CSVs of predictions and probabilities.

Two scenarios are supported:
- `drop` — exclude FL_UDSD == 2 (Impaired Not SCD/MCI), keep 4 classes
- `merge` — fold FL_UDSD == 2 into 4 (EMCI), keep 4 classes

## Configuration

In [ ]:
# ---- Edit these for each run ----
GROUPS_TO_USE   = ["CDRSUM","HVLTDR", "PLASMA", "APOE"]   # pick from SELECTABLE_GROUPS below
MODEL_NAME      = "xgb"                         # "lr" | "rf" | "xgb"
SCENARIO        = "drop"                        # "drop" | "merge"
DATA_PATH       = "../data/clinical/preprocess.csv"
N_SPLITS        = 4
SAVE_PREDICTIONS = True
RESULTS_DIR     = "results/cm"

## Setup

Thread-pool caps must be set **before** numpy/sklearn/xgboost are imported, so this cell does that first. If you re-run after changing `OMP_NUM_THREADS` etc., you'll need to restart the kernel.

In [ ]:
import os
for var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
            "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(var, "1")

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedGroupKFold

import xgboost as xgb

warnings.filterwarnings("ignore", category=UserWarning)

## Feature groups

These are the groups you can choose from in `GROUPS_TO_USE`. Edit here if you want to add a new group or change an existing one.

In [ ]:
MRI_VOLUME = [
    "VOL_ENTRHNA_L", "VOL_ENTRHNA_R", "VOL_HIPP_L", "VOL_HIPP_R",
    "VOL_AMYG_L", "VOL_AMYG_R", "VOL_PRECUNE_L", "VOL_PRECUNE_R",
    "VOL_POST_CING_L", "VOL_POST_CING_R", "VOL_INF_PAR_L", "VOL_INF_PAR_R",
    "VOL_INF_TEMP_L", "VOL_INF_TEMP_R", "VOL_TEMP_PL_L", "VOL_TEMP_PL_R",
    "VOL_LAT_ORB_L", "VOL_LAT_ORB_R", "VOL_SUP_FRNT_L", "VOL_SUP_FRNT_R",
    "VOL_PRECENT_L", "VOL_PRECENT_R",
    "VOL_HIPP_SBCLM_HEAD_L", "VOL_HIPP_SBCLM_HEAD_R",
    "VOL_HIPP_SBCLM_BOD_L", "VOL_HIPP_SBCLM_BOD_R",
    "VOL_HIPP_PRESBCLM_HEAD_L", "VOL_HIPP_PRESBCLM_HEAD_R",
    "VOL_HIPP_PRESBCLM_BOD_L", "VOL_HIPP_PRESBCLM_BOD_R",
    "VOL_HIPP_CA1_HEAD_L", "VOL_HIPP_CA1_HEAD_R",
    "VOL_HIPP_CA1_BOD_L", "VOL_HIPP_CA1_BOD_R",
]

MRI_THICKNESS = [
    "THK_ENTRHNA_L", "THK_ENTRHNA_R", "THK_PARAHIPP_L", "THK_PARAHIPP_R",
    "THK_PRECUNE_L", "THK_PRECUNE_R", "THK_POST_CING_L", "THK_POST_CING_R",
    "THK_INF_PAR_L", "THK_INF_PAR_R", "THK_INF_TEMP_L", "THK_INF_TEMP_R",
    "THK_TEMP_PL_L", "THK_TEMP_PL_R", "THK_LAT_ORB_L", "THK_LAT_ORB_R",
    "THK_ROST_MIDFRNT_L", "THK_ROST_MIDFRNT_R",
    "THK_CAUD_MIDFRNT_L", "THK_CAUD_MIDFRNT_R",
    "THK_SUP_FRNT_L", "THK_SUP_FRNT_R", "THK_PRECENT_L", "THK_PRECENT_R",
]

FEATURE_GROUPS = {
    "INFO":           ["PTID", "VISITYR", "NACCAGE", ""],
    "CDRSUM":         ["CDRSUM"],
    "MMSE":           ["MMSE"],
    "HVLTDR":         ["HVLT_DR"],
    "LASSI":          ["LASSI_A_CR2", "LASSI_A_CR2_INT","LASSI_B_CR1", "LASSI_B_CR1_INT", "LASSI_B_CR2", "LASSI_B_CR2_INT"],
    "PLASMA":         ["PTAU_217_CONCNTRTN"],
    "APOE":           ["APOE4S"],
    "MRI":            MRI_VOLUME + MRI_THICKNESS,
    "TARGETS":        ["FL_UDSD", "NACCETPR"],
}


In [ ]:

print("Available feature groups:")
for name, cols in FEATURE_GROUPS.items():
    preview = ", ".join(cols[:4])
    more = f" ... (+{len(cols)-4} more)" if len(cols) > 4 else ""
    print(f"  {name:10s} ({len(cols):>2} feature{'s' if len(cols)!=1 else ''}): {preview}{more}")

## Helpers

In [ ]:
def make_model(model_name):
    if model_name == "lr":
        return LogisticRegression(
            max_iter=5000, random_state=42, class_weight="balanced", n_jobs=1,
        )
    if model_name == "rf":
        return RandomForestClassifier(
            n_estimators=250, random_state=42, class_weight="balanced", n_jobs=1,
        )
    if model_name == "xgb":
        return xgb.XGBClassifier(
            n_estimators=100,
            random_state=42,
            objective="multi:softprob",
            eval_metric="mlogloss",
            tree_method="hist",
            n_jobs=1,
        )
    raise ValueError(f"Unknown model {model_name!r}")


def prepare_scenario(df, scenario):
    """Returns (df_prepared, label_map). df_prepared has FL_UDSD_CAT as the integer target."""
    if scenario == "drop":
        sub = df[df["FL_UDSD"] != 2].copy()
        diagnosis_order = ["NC", "SCD",
                           "EMCI", "LMCI", "Dementia"]
    elif scenario == "merge":
        sub = df.copy()
        sub.loc[sub["FL_UDSD"] == 2, "FL_UDSD"] = 4
        diagnosis_order = ["Normal cognition", "SCD",
                           "Early Impaired", "LMCI", "Dementia"]
    else:
        raise ValueError(f"Unknown scenario: {scenario}")

    mapping = {old: new for new, old in enumerate(sorted(sub["FL_UDSD"].unique()))}
    sub["FL_UDSD_CAT"] = sub["FL_UDSD"].map(mapping)
    label_map = {i: diagnosis_order[i] for i in sorted(sub["FL_UDSD_CAT"].unique())}
    return sub, label_map


def print_confusion(cm, class_names, indent=""):
    """Pretty-print a confusion matrix with row/col labels."""
    name_w = max(len(n) for n in class_names)
    cell_w = max(6, len(str(cm.max())) + 2)
    header = " " * (name_w + 2) + "".join(f"{n[:cell_w-1]:>{cell_w}}" for n in class_names)
    print(indent + header)
    for name, row in zip(class_names, cm):
        line = f"{name:>{name_w}}  " + "".join(f"{v:>{cell_w}}" for v in row)
        print(indent + line)

## Build features and load data

In [ ]:
# Validate group selection
unknown = [g for g in GROUPS_TO_USE if g not in FEATURE_GROUPS]
if unknown:
    raise ValueError(f"Unknown feature group(s): {unknown}. "
                     f"Valid: {sorted(FEATURE_GROUPS.keys())}")

# Build feature list (preserve order, dedupe)
seen = set()
features = []
for g in GROUPS_TO_USE:
    for col in FEATURE_GROUPS[g]:
        if col not in seen:
            features.append(col)
            seen.add(col)

print("=" * 72)
print("Single-experiment validation")
print("=" * 72)
print(f"Model        : {MODEL_NAME}")
print(f"Scenario     : {SCENARIO}")
print(f"Feature grps : {GROUPS_TO_USE}")
print(f"Total feats  : {len(features)}")
print(f"Data         : {DATA_PATH}")

data_path = Path(DATA_PATH)
if not data_path.exists():
    raise FileNotFoundError(f"data file not found: {data_path}")

df = pd.read_csv(data_path)

missing = [c for c in features + ["PTID", "FL_UDSD"] if c not in df.columns]
if missing:
    raise ValueError(f"columns missing from data: {missing}")

df_prepared, label_map = prepare_scenario(df, SCENARIO)

X = df_prepared[features]
y = df_prepared["FL_UDSD_CAT"]
groups = df_prepared["PTID"]

classes_int = sorted(y.unique())
class_names = [label_map[c] for c in classes_int]

print("\nClass index -> label:")
for i, name in label_map.items():
    n = int((y == i).sum())
    print(f"  {i}: {name:32s}  n={n}")

print(f"\nTotal samples: {len(y)}  |  unique groups (PTID): {groups.nunique()}")
print(f"Features: {X.shape[1]}")

## Run cross-validation

We walk folds manually (instead of using `cross_validate`) so we can collect predictions, probabilities, and confusion matrices per fold, and verify there's no group leakage at every split.

In [ ]:
cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
model = make_model(MODEL_NAME)

fold_summaries = []
all_predictions = []
confusion_per_fold = []
fold_models = []

for fold_i, (tr_idx, te_idx) in enumerate(cv.split(X, y, groups), start=1):
    X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
    y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]
    g_tr, g_te = groups.iloc[tr_idx], groups.iloc[te_idx]

    # Group-leakage sanity check
    leak = set(g_tr).intersection(set(g_te))
    if leak:
        print(f"  [FOLD {fold_i}] WARNING: {len(leak)} PTID(s) in both train and test!")

    model.fit(X_tr, y_tr)
    fold_models.append(model)
    model = make_model(MODEL_NAME)
    y_pred = fold_models[-1].predict(X_te)
    try:
        y_proba = fold_models[-1].predict_proba(X_te)
    except AttributeError:
        y_proba = None

    cm = confusion_matrix(y_te, y_pred, labels=classes_int)
    confusion_per_fold.append(cm)

    f1m = f1_score(y_te, y_pred, average="macro", zero_division=0)
    bal = balanced_accuracy_score(y_te, y_pred)

    fold_summaries.append({
        "fold": fold_i,
        "n_train": len(tr_idx),
        "n_test": len(te_idx),
        "n_train_groups": g_tr.nunique(),
        "n_test_groups": g_te.nunique(),
        "f1_macro": f1m,
        "balanced_accuracy": bal,
        "leaked_groups": len(leak),
    })

    print(f"=== Fold {fold_i} ===")
    print(f"  train: {len(tr_idx)} samples / {g_tr.nunique()} groups   "
          f"test: {len(te_idx)} samples / {g_te.nunique()} groups")
    print(f"  f1_macro={f1m:.4f}   balanced_accuracy={bal:.4f}")
    print()
    print(classification_report(
        y_te, y_pred,
        labels=classes_int,
        target_names=class_names,
        zero_division=0,
        digits=4,
    ))
    print("  Confusion matrix (rows=true, cols=pred):")
    print_confusion(cm, class_names, indent="    ")
    print()

    if SAVE_PREDICTIONS:
        rec = pd.DataFrame({
            "fold": fold_i,
            "PTID": g_te.values,
            "y_true": y_te.values,
            "y_true_label": [label_map[v] for v in y_te.values],
            "y_pred": y_pred,
            "y_pred_label": [label_map[v] for v in y_pred],
        })
        if y_proba is not None:
            # XGBoost may return probas in a different class order; align it.
            model_classes = list(getattr(fold_models[-1], "classes_", classes_int))
            for c_int, c_name in zip(classes_int, class_names):
                if c_int in model_classes:
                    col = model_classes.index(c_int)
                    rec[f"proba_{c_name}"] = y_proba[:, col]
                else:
                    rec[f"proba_{c_name}"] = np.nan
        all_predictions.append(rec)

fold_summaries_df = pd.DataFrame(fold_summaries)
summed_cm = np.sum(confusion_per_fold, axis=0)
predictions_df = pd.concat(all_predictions, ignore_index=True) if all_predictions else None

## Overall summary

In [ ]:
print("=" * 72)
print("OVERALL SUMMARY")
print("=" * 72)
print(fold_summaries_df.to_string(index=False))
print()
print(f"f1_macro:           "
      f"mean={fold_summaries_df['f1_macro'].mean():.4f}  "
      f"std={fold_summaries_df['f1_macro'].std(ddof=0):.4f}")
print(f"balanced_accuracy:  "
      f"mean={fold_summaries_df['balanced_accuracy'].mean():.4f}  "
      f"std={fold_summaries_df['balanced_accuracy'].std(ddof=0):.4f}")

print("\nSummed confusion matrix (rows=true, cols=pred):")
print_confusion(summed_cm, class_names, indent="  ")

# Row-normalized = recall-per-class view
cm = summed_cm.astype(float)
row_sums = cm.sum(axis=1, keepdims=True)
cm_norm = np.divide(cm, row_sums, out=np.zeros_like(cm), where=row_sums != 0)
print("\nRow-normalized (recall view, %):")
cm_pct = np.round(cm_norm * 100, 1)
print_confusion(cm_pct, class_names, indent="  ")

In [ ]:
INSPECT_FOLD = 2  # 1-indexed; None = summed across all folds

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
from pathlib import Path

cm_to_show = confusion_per_fold[INSPECT_FOLD - 1] if INSPECT_FOLD else summed_cm
fold_tag = f"fold {INSPECT_FOLD}" if INSPECT_FOLD else f"{N_SPLITS}-fold summed"

fig, axes = plt.subplots(1, 1, figsize=(16, 6))

disp_raw = ConfusionMatrixDisplay(confusion_matrix=cm_to_show, display_labels=class_names)
disp_raw.plot(ax=axes, colorbar=False, cmap="Blues", xticks_rotation=90)
axes.set_title(f"Confusion matrix — {MODEL_NAME.upper()} ({fold_tag}) - {GROUPS_TO_USE}")
axes.set_xlabel("Predicted label")
axes.set_ylabel("True label")

plt.tight_layout()
plt.show()

In [ ]:
# INSPECT_FOLD = 2  # 1-indexed; None = summed across all folds

# import matplotlib.pyplot as plt
# from sklearn.metrics import ConfusionMatrixDisplay
# from pathlib import Path

# cm_to_show = confusion_per_fold[INSPECT_FOLD - 1] if INSPECT_FOLD else summed_cm
# fold_tag = f"fold {INSPECT_FOLD}" if INSPECT_FOLD else f"{N_SPLITS}-fold summed"

# fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# disp_raw = ConfusionMatrixDisplay(confusion_matrix=cm_to_show, display_labels=class_names)
# disp_raw.plot(ax=axes[0], colorbar=False, cmap="Blues", xticks_rotation=90)
# axes[0].set_title(f"Confusion matrix — {MODEL_NAME.upper()} / {SCENARIO} ({fold_tag})")
# axes[0].set_xlabel("Predicted label")
# axes[0].set_ylabel("True label")

# cm_norm = cm_to_show.astype(float) / cm_to_show.sum(axis=1, keepdims=True)
# disp_norm = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=class_names)
# disp_norm.plot(ax=axes[1], colorbar=False, cmap="Blues", xticks_rotation=45)
# axes[1].set_title(f"Row-normalised (recall view) — {fold_tag}")
# axes[1].set_xlabel("Predicted label")
# axes[1].set_ylabel("True label")
# for text in axes[1].texts:
#     val = float(text.get_text())
#     text.set_text(f"{val:.0%}")

# plt.tight_layout()
# plt.show()

In [ ]:
results_dir = Path(RESULTS_DIR)
results_dir.mkdir(parents=True, exist_ok=True)
tag = f"{MODEL_NAME}_{SCENARIO}_{'_'.join(GROUPS_TO_USE)}"

fold_items = [(i + 1, cm, f"fold{i + 1}") for i, cm in enumerate(confusion_per_fold)]
fold_items.append((None, summed_cm, f"{N_SPLITS}fold_summed"))

for fold_num, cm_to_show, fold_tag in fold_items:
    fig, axes = plt.subplots(1, 1, figsize=(16, 6))

    disp_raw = ConfusionMatrixDisplay(confusion_matrix=cm_to_show, display_labels=class_names)
    disp_raw.plot(ax=axes, colorbar=False, cmap="Blues", xticks_rotation=90)
    axes.set_title(f"Confusion matrix — {MODEL_NAME.upper()} - ({fold_tag} -{GROUPS_TO_USE})")
    axes.set_xlabel("Predicted label")
    axes.set_ylabel("True label")



    plt.tight_layout()
    out_path = results_dir / f"confusion_{tag}_{fold_tag}.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out_path}")

In [ ]:
# results_dir = Path(RESULTS_DIR)
# results_dir.mkdir(parents=True, exist_ok=True)
# tag = f"{MODEL_NAME}_{SCENARIO}_{'_'.join(GROUPS_TO_USE)}"

# fold_items = [(i + 1, cm, f"fold{i + 1}") for i, cm in enumerate(confusion_per_fold)]
# fold_items.append((None, summed_cm, f"{N_SPLITS}fold_summed"))

# for fold_num, cm_to_show, fold_tag in fold_items:
#     fig, axes = plt.subplots(1, 2, figsize=(16, 6))

#     disp_raw = ConfusionMatrixDisplay(confusion_matrix=cm_to_show, display_labels=class_names)
#     disp_raw.plot(ax=axes[0], colorbar=False, cmap="Blues", xticks_rotation=90)
#     axes[0].set_title(f"Confusion matrix — {MODEL_NAME.upper()} / ({fold_tag} {GROUPS_TO_USE})")
#     axes[0].set_xlabel("Predicted label")
#     axes[0].set_ylabel("True label")

#     cm_norm = cm_to_show.astype(float) / cm_to_show.sum(axis=1, keepdims=True)
#     disp_norm = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=class_names)
#     disp_norm.plot(ax=axes[1], colorbar=False, cmap="Blues", xticks_rotation=45)
#     axes[1].set_title(f"Row-normalised (recall view) — {fold_tag}")
#     axes[1].set_xlabel("Predicted label")
#     axes[1].set_ylabel("True label")
#     for text in axes[1].texts:
#         val = float(text.get_text())
#         text.set_text(f"{val:.0%}")

#     plt.tight_layout()
#     out_path = results_dir / f"confusion_{tag}_{fold_tag}.png"
#     fig.savefig(out_path, dpi=150, bbox_inches="tight")
#     plt.close(fig)
#     print(f"Saved: {out_path}")

In [ ]:
TOP_N_FEATURES = 50  # set to None to show all

if not fold_models or not hasattr(fold_models[0], "feature_importances_"):
    print(f"{MODEL_NAME!r} does not expose feature_importances_")
else:
    if INSPECT_FOLD is None:
        imp_values = np.mean([m.feature_importances_ for m in fold_models], axis=0)
        title = f"Feature importances — {MODEL_NAME.upper()} (mean over {N_SPLITS} folds)"
    else:
        imp_values = fold_models[INSPECT_FOLD - 1].feature_importances_
        title = f"Feature importances — {MODEL_NAME.upper()} (fold {INSPECT_FOLD})"

    importances = pd.Series(imp_values, index=features).sort_values(ascending=False)
    if TOP_N_FEATURES is not None:
        importances = importances.head(TOP_N_FEATURES)
        title += f" — top {len(importances)}"
    importances = importances.sort_values(ascending=True)  # flip for horizontal bar

    fig, ax = plt.subplots(figsize=(7, max(3, len(importances) * 0.4)))
    importances.plot.barh(ax=ax)
    ax.set_xlabel("Importance")
    ax.set_title(title)
    ax.bar_label(ax.containers[0], fmt="%.3f", padding=3)
    plt.tight_layout()
    plt.show()

## Save outputs

In [ ]:
results_dir = Path(RESULTS_DIR)
results_dir.mkdir(parents=True, exist_ok=True)
tag = f"{MODEL_NAME}_{SCENARIO}_{'_'.join(GROUPS_TO_USE)}"

fold_path = results_dir / f"validation_{tag}_folds.csv"
fold_summaries_df.to_csv(fold_path, index=False)
print(f"Saved per-fold summary: {fold_path}")

cm_path = results_dir / f"validation_{tag}_confusion.csv"
pd.DataFrame(
    summed_cm,
    index=class_names,
    columns=class_names,
).to_csv(cm_path)
print(f"Saved summed confusion: {cm_path}")

if SAVE_PREDICTIONS and predictions_df is not None:
    pred_path = results_dir / f"validation_{tag}_predictions.csv"
    predictions_df.to_csv(pred_path, index=False)
    print(f"Saved predictions:      {pred_path}")

## Inspect predictions

Quick views into the predictions DataFrame for further exploration. Skip if `SAVE_PREDICTIONS = False`.

In [ ]:
# if predictions_df is not None:
#     display(predictions_df.head(10)
#             .style.format({c: lambda v: f"{v * 100:.4g}%" for c in proba_cols}))

In [ ]:
# Misclassifications only — useful for spot-checking which patients are hard
# Set INSPECT_FOLD (defined above) to a fold number to filter, or None for all folds
INSPECT_FOLD=3
if predictions_df is not None:
    if INSPECT_FOLD is not None:
        subset = predictions_df[predictions_df["fold"] == INSPECT_FOLD].copy()
        fold_label = f"fold {INSPECT_FOLD}"
    else:
        subset = predictions_df.copy()
        fold_label = "all folds"

    misses = (
        subset[subset["y_true"] != subset["y_pred"]]
        .sort_values(by=["y_true_label","y_pred_label"], ascending=False)
        .reset_index(drop=True)
    )
    print(f"{len(misses)} of {len(subset)} predictions are wrong "
          f"({len(misses) / len(subset):.1%})  [{fold_label}]")

    proba_cols = [c for c in misses.columns if c.startswith("proba_")]
    display_cols = ["fold", "y_true_label", "y_pred_label"] + proba_cols

    fmt_df = misses[display_cols].copy()
    for c in proba_cols:
        fmt_df[c] = fmt_df[c].map(lambda v: f"{v * 100:.2f}%")

    col_labels = display_cols
    cell_text = fmt_df[col_labels].values.tolist()

    fig, ax = plt.subplots(figsize=(10,5))
    ax.axis("off")

    tbl = ax.table(
        cellText=cell_text,
        colLabels=col_labels,
        loc="center",
        cellLoc="center",
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.auto_set_column_width(range(len(col_labels)))

    for col_i in range(len(col_labels)):
        tbl[0, col_i].set_facecolor("#2c3e50")
        tbl[0, col_i].set_text_props(color="white", fontweight="bold")

    for row_i in range(1, len(misses) + 1):
        colour = "#f2f2f2" if row_i % 2 == 0 else "white"
        for col_i in range(len(col_labels)):
            tbl[row_i, col_i].set_facecolor(colour)

    ax.set_title(
        f"Misclassifications — {MODEL_NAME.upper()} [{fold_label}]  "
        f"({len(misses)}/{len(subset)} shown, {len(misses)/len(subset):.1%}) \n {GROUPS_TO_USE}",
        fontsize=11, fontweight="bold", pad=12
    )
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

In [ ]:
# Where do the errors land? Confusion as a crosstab
if predictions_df is not None:
    display(pd.crosstab(
        predictions_df["y_true_label"],
        predictions_df["y_pred_label"],
        rownames=["true"], colnames=["pred"],
    ))